In [ ]:
import scanpy as sc
import rpy2.robjects as ro
from scipy.stats import median_abs_deviation as mad
from pathlib import Path
from src import *

%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

sc.set_figure_params(dpi=150, facecolor='white')
sc.settings.verbosity = 3

project_root = Path.cwd().parent
data_dir = project_root / "data"
results_dir = project_root / "results"
ro.r(f'renv::load("{project_root}")')

In [ ]:
immune = sc.read_h5ad(data_dir / "processed" / "01_immune.h5ad")
nk_ilc = immune[immune.obs["celltype"] == "nks/ilcs"].copy()

In [ ]:
nk_ilc = deviant_hvg(nk_ilc)

In [ ]:
plot_deviance(nk_ilc)

In [ ]:
harmony_embed(nk_ilc)
sc.pl.pca_overview(nk_ilc, color="sample_id")

In [ ]:
cluster_subset(nk_ilc, n_pcs=15, resolution=0.5, min_dist=0.75)

In [ ]:
sc.pl.umap(nk_ilc, color=["scDblFinder_score", "sample_id", "pct_counts_mt", "leiden"], ncols=2, size=200)

In [ ]:
nk_ilc_markers = {
    "nks": ["Ncr1", "Eomes", "Klrb1c", "Nkg7"],
    "ilc2s": ["Gata3", "Rora", "Il1rl1", "Areg"],
    "ilc3s": ["Rorc", "Il7r"],
    "gd_t_cells": ["Trdc", "Sox13", "Trgc1", "Trgc2"]
}

sc.tl.dendrogram(nk_ilc, groupby="leiden", use_rep="X_pca_harmony")
sc.pl.matrixplot(
    nk_ilc,
    var_names=nk_ilc_markers,
    groupby="leiden",
    standard_scale="var",
    dendrogram=True,
    cmap="Reds"
)
sc.pl.umap(nk_ilc, color=["Ncr1", "Eomes", "Nkg7", "Itgam", "Cd27"], size=300, vmax="p99", ncols=3)

In [ ]:
nk_ilc_celltypes = {
    "nks": ["3", "4"],
    "ilc2s": ["1"],
    "ilc3s": ["2"],
    "gd_t_cells": ["0"]
}

apply_celltypes(nk_ilc, nk_ilc_celltypes)

In [ ]:
sc.pl.umap(nk_ilc, color="celltype", size=200)

In [ ]:
if "celltype" not in immune.obs:
    immune.obs["celltype"] = "unannotated"
immune.obs["celltype"] = immune.obs["celltype"].astype(str)

In [ ]:
immune.obs.loc[nk_ilc.obs_names, "celltype"] = nk_ilc.obs["celltype"].astype(str).values

In [ ]:
nk_ilc_final = nk_ilc[nk_ilc.obs["celltype"].isin(["nks", "ilc2s", "ilc3s"])].copy()

In [ ]:
sc.pl.umap(nk_ilc_final, color="celltype", size=300)

In [ ]:
immune.write_h5ad(data_dir / "processed" / "01_immune.h5ad")
nk_ilc_final.write_h5ad(data_dir / "processed" / "05_nk_ilc.h5ad")